In [1]:

import os, json, time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

CSV_PATH = Path("classification_with_c110_d110_errors_snr.csv")

OUT_DIR = Path.cwd() / "logreg_c110_truncation_out"
RUNS_DIR = OUT_DIR / "runs"
REPORT_DIR = OUT_DIR / "analysis_report"
for d in [OUT_DIR, RUNS_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HEARTBEAT_PATH = OUT_DIR / "HEARTBEAT.json"
LAST_EVENT_PATH = OUT_DIR / "LAST_EVENT.txt"
MANIFEST_PATH = OUT_DIR / "RUNNING_MANIFEST.json"

def hb(payload):
    payload = dict(payload)
    payload["time"] = time.ctime()
    HEARTBEAT_PATH.write_text(json.dumps(payload, indent=2))

def log_event(msg):
    LAST_EVENT_PATH.write_text(f"[{time.ctime()}] {msg}\n")

# ------------------------
# Experiment config
# ------------------------
K_MIN, K_MAX = 1, 55
K_LIST = list(range(K_MIN, K_MAX + 1))

N_REPEATS = 100
TEST_FRAC = 0.15
VAL_FRAC = 0.15
VAL_FRAC_OF_REMAIN = VAL_FRAC / (1.0 - TEST_FRAC)

BASE_SEED = 1234
THR_GRID_SIZE = 800

LR_PARAMS = dict(
    penalty="l2",
    C=1.0,
    solver="saga",
    class_weight="balanced",
    max_iter=5000,
    tol=1e-4,
    n_jobs=-1,
    random_state=0,
)

manifest = dict(
    started_at=time.ctime(),
    csv=str(CSV_PATH),
    out_dir=str(OUT_DIR.resolve()),
    plan=dict(
        K_MIN=K_MIN, K_MAX=K_MAX, N_REPEATS=N_REPEATS,
        TEST_FRAC=TEST_FRAC, VAL_FRAC=VAL_FRAC,
        THR_GRID_SIZE=THR_GRID_SIZE,
        LR_PARAMS=LR_PARAMS,
    ),
)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
log_event("Manifest created.")
hb({"status": "ready"})

print("OUT_DIR:", OUT_DIR.resolve())

/Users/kris/Desktop/Astrophysics/.venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OUT_DIR: /Users/kris/Desktop/Astrophysics/logreg_c110_truncation_out


In [2]:
df = pd.read_csv(CSV_PATH)
if "y" not in df.columns:
    raise KeyError("CSV must contain 'y'")

df["y"] = df["y"].astype(int)

C110 = [f"c{i:03d}" for i in range(110)]
missing = [c for c in C110 if c not in df.columns]
if missing:
    raise KeyError(f"Missing c110 columns, e.g. {missing[:6]}")

# c000..c054 = BP, c055..c109 = RP
BP = [f"c{i:03d}" for i in range(55)]
RP = [f"c{i:03d}" for i in range(55, 110)]

Xbp = df[BP].to_numpy(np.float32)
Xrp = df[RP].to_numpy(np.float32)
y_all = df["y"].to_numpy(int)

print("Xbp:", Xbp.shape, "Xrp:", Xrp.shape, "pos_rate:", float(y_all.mean()))

Xbp: (2815, 55) Xrp: (2815, 55) pos_rate: 0.19822380106571935


In [3]:
def make_splits(y, n_repeats, base_seed):
    splitter = StratifiedShuffleSplit(n_splits=n_repeats, test_size=TEST_FRAC, random_state=base_seed)
    idx_all = np.arange(len(y))
    splits = []
    for r, (trainval_rel, test_rel) in enumerate(splitter.split(np.zeros(len(y)), y)):
        idx_trainval = idx_all[trainval_rel]
        idx_test = idx_all[test_rel]

        y_tv = y[idx_trainval]
        splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRAC_OF_REMAIN, random_state=base_seed + 1000 + r)
        tr_rel2, va_rel2 = next(splitter2.split(np.zeros(len(y_tv)), y_tv))

        idx_train = idx_trainval[tr_rel2]
        idx_val   = idx_trainval[va_rel2]
        splits.append(dict(repeat=r, train_idx=idx_train, val_idx=idx_val, test_idx=idx_test))
    return splits

splits = make_splits(y_all, N_REPEATS, BASE_SEED)
print("splits:", len(splits), "sizes:", {k: len(splits[0][k]) for k in ["train_idx","val_idx","test_idx"]})

splits: 100 sizes: {'train_idx': 1969, 'val_idx': 423, 'test_idx': 423}


In [4]:
def make_X_for_K(idx, K):
    bp = Xbp[idx, :K]   # (N,K)
    rp = Xrp[idx, :K]   # (N,K)
    X = np.concatenate([bp, rp], axis=1).astype(np.float32, copy=False)  # (N, 2K)
    return X

Xdemo = make_X_for_K(np.arange(5), 10)
print("demo shape:", Xdemo.shape)

demo shape: (5, 20)


In [5]:
def prob_metrics(y_true, p):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    pc = np.clip(p, 1e-15, 1-1e-15)
    return dict(
        ROC_AUC=roc_auc_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        PR_AUC=average_precision_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        Brier=brier_score_loss(y_true, p),
        LogLoss=log_loss(y_true, pc, labels=[0,1]),
    )

def confusion_metrics(y_true, y_hat):
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if (tp+fn) else np.nan
    spec = tn/(tn+fp) if (tn+fp) else np.nan
    prec = tp/(tp+fp) if (tp+fp) else np.nan
    return prec, sens, spec, tn, fp, fn, tp

def threshold_grid(p, n=800):
    lo, hi = float(np.min(p)), float(np.max(p))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return np.array([0.5])
    return np.linspace(lo, hi, n)

def choose_threshold(y_true, p, criterion="youden", n=800):
    best_thr, best_score = 0.5, -np.inf
    for thr in threshold_grid(p, n):
        yhat = (p >= thr).astype(int)
        prec, sens, spec, *_ = confusion_metrics(y_true, yhat)
        if criterion == "youden":
            score = (sens + spec - 1) if np.isfinite(sens) and np.isfinite(spec) else -np.inf
        elif criterion == "f1":
            score = (2*prec*sens/(prec+sens)) if np.isfinite(prec) and np.isfinite(sens) and (prec+sens)>0 else -np.inf
        else:
            raise ValueError
        if score > best_score:
            best_score, best_thr = float(score), float(thr)
    return best_thr, float(best_score)

def evaluate(yv, pv, yt, pt):
    out = {f"test_{k}": float(v) for k,v in prob_metrics(yt, pt).items()}
    for crit in ["youden", "f1"]:
        thr, sc = choose_threshold(yv, pv, crit, THR_GRID_SIZE)
        yhat = (pt >= thr).astype(int)
        prec, sens, spec, tn, fp, fn, tp = confusion_metrics(yt, yhat)
        out[f"{crit}_val_thr"] = float(thr)
        out[f"{crit}_val_score"] = float(sc)
        out[f"{crit}_test_Precision"] = float(prec)
        out[f"{crit}_test_Sensitivity"] = float(sens)
        out[f"{crit}_test_Specificity"] = float(spec)
        out[f"{crit}_test_tn"] = int(tn); out[f"{crit}_test_fp"] = int(fp)
        out[f"{crit}_test_fn"] = int(fn); out[f"{crit}_test_tp"] = int(tp)
    return out

In [6]:
def run_id(K, rep):
    return f"K{int(K):02d}__r{int(rep):02d}"

def shard_path(K, rep):
    return RUNS_DIR / f"{run_id(K, rep)}.parquet"

def done(K, rep):
    return shard_path(K, rep).exists()

def save_row(row):
    pd.DataFrame([row]).to_parquet(RUNS_DIR / f"{row['run_id']}.parquet", index=False)

def load_all_rows():
    files = sorted(RUNS_DIR.glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

print("Existing shards:", len(list(RUNS_DIR.glob("*.parquet"))))


Existing shards: 5500


In [7]:
pK = tqdm(K_LIST, desc="K (coeffs per BP/RP)", leave=True)

for K in pK:
    pK.set_postfix_str(f"K={K}")
    log_event(f"Starting K={K}")

    p_rep = tqdm(splits, desc=f"Repeats [K={K}]", leave=False)
    for sp in p_rep:
        rep = sp["repeat"]
        if done(K, rep):
            continue

        hb({"status": "run_start", "K": int(K), "repeat": int(rep)})

        tr, va, te = sp["train_idx"], sp["val_idx"], sp["test_idx"]

        Xtr = make_X_for_K(tr, K); ytr = y_all[tr]
        Xva = make_X_for_K(va, K); yva = y_all[va]
        Xte = make_X_for_K(te, K); yte = y_all[te]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xva = scaler.transform(Xva)
        Xte = scaler.transform(Xte)

        params = dict(LR_PARAMS)
        params["random_state"] = BASE_SEED + rep

        model = LogisticRegression(**params)

        t0 = time.time()
        model.fit(Xtr, ytr)

        pv = model.predict_proba(Xva)[:, 1]
        pt = model.predict_proba(Xte)[:, 1]

        met = evaluate(yva, pv, yte, pt)
        dt = time.time() - t0

        row = dict(
            run_id=run_id(K, rep),
            K=int(K),
            repeat=int(rep),
            runtime_s=float(dt),
            n_train=int(len(ytr)),
            n_val=int(len(yva)),
            n_test=int(len(yte)),
            test_pos_rate=float(np.mean(yte)),
            **met
        )
        save_row(row)

        hb({"status": "run_done", "K": int(K), "repeat": int(rep), "runtime_s": dt})

log_event("ALL DONE.")
hb({"status": "completed"})
print("Done. Shards:", len(list(RUNS_DIR.glob('*.parquet'))))


K (coeffs per BP/RP): 100%|██████████| 55/55 [00:00<00:00, 392.28it/s, K=55]

Done. Shards: 5500


In [8]:
res = load_all_rows()
print("rows:", res.shape)
display(res.head(5))

res.to_parquet(OUT_DIR / "truncation_logreg_raw.parquet", index=False)
res.to_csv(OUT_DIR / "truncation_logreg_raw.csv", index=False)
print("Saved raw outputs.")

rows: (5500, 30)


,run_id,K,repeat,runtime_s,n_train,n_val,n_test,test_pos_rate,test_ROC_AUC,test_PR_AUC,...,youden_test_tp,f1_val_thr,f1_val_score,f1_test_Precision,f1_test_Sensitivity,f1_test_Specificity,f1_test_tn,f1_test_fp,f1_test_fn,f1_test_tp
0,K01__r00,1,0,0.338783,1969,423,423,0.198582,0.926464,0.800454,...,72,0.430618,0.736842,0.757895,0.857143,0.932153,316,23,12,72
1,K01__r01,1,1,0.312793,1969,423,423,0.198582,0.887976,0.770684,...,69,0.451358,0.819876,0.708333,0.809524,0.917404,311,28,16,68
2,K01__r02,1,2,0.358218,1969,423,423,0.198582,0.834843,0.707645,...,57,0.545699,0.795181,0.702703,0.619048,0.935103,317,22,32,52
3,K01__r03,1,3,0.344165,1969,423,423,0.198582,0.916210,0.782587,...,73,0.464103,0.748663,0.717391,0.785714,0.923304,313,26,18,66
4,K01__r04,1,4,0.319864,1969,423,423,0.198582,0.930327,0.815436,...,67,0.510343,0.800000,0.761364,0.797619,0.938053,318,21,17,67


Saved raw outputs.


In [9]:
metrics = [
    "runtime_s",
    "test_PR_AUC","test_ROC_AUC","test_LogLoss","test_Brier",
    "youden_test_Precision","youden_test_Sensitivity","youden_test_Specificity","youden_val_thr",
    "f1_test_Precision","f1_test_Sensitivity","f1_test_Specificity","f1_val_thr"
]

summ = res.groupby("K", as_index=False)[metrics].agg(["mean","std"])
summ.columns = [c if s == "" else f"{c}_{s}" for c, s in summ.columns]
summ = summ.sort_values("K")

summ.to_csv(OUT_DIR / "truncation_logreg_summary_byK.csv", index=False)
display(summ.head(10))

,K,runtime_s_mean,runtime_s_std,test_PR_AUC_mean,test_PR_AUC_std,test_ROC_AUC_mean,test_ROC_AUC_std,test_LogLoss_mean,test_LogLoss_std,test_Brier_mean,...,youden_val_thr_mean,youden_val_thr_std,f1_test_Precision_mean,f1_test_Precision_std,f1_test_Sensitivity_mean,f1_test_Sensitivity_std,f1_test_Specificity_mean,f1_test_Specificity_std,f1_val_thr_mean,f1_val_thr_std
0,1,0.328563,0.008946,0.784757,0.040961,0.898366,0.023966,0.381980,0.029201,0.100447,...,0.409502,0.048786,0.737563,0.050995,0.766310,0.061839,0.930826,0.020784,0.504505,0.072727
1,2,0.353207,0.007391,0.868814,0.030728,0.912502,0.023094,0.280171,0.020541,0.071266,...,0.489728,0.088476,0.873230,0.045740,0.782024,0.058043,0.970914,0.012745,0.638448,0.111287
2,3,0.352055,0.007891,0.880971,0.029497,0.926613,0.019861,0.265443,0.020688,0.067568,...,0.500410,0.122145,0.884885,0.042842,0.794643,0.050320,0.973717,0.011658,0.657175,0.095989
3,4,0.357948,0.009378,0.879607,0.029772,0.924704,0.020786,0.266567,0.021040,0.067810,...,0.503168,0.116941,0.889799,0.042319,0.791548,0.049549,0.975133,0.011103,0.667787,0.092967
4,5,0.367666,0.008739,0.878825,0.029838,0.925990,0.020626,0.264723,0.021883,0.067085,...,0.488989,0.125830,0.881891,0.045648,0.792500,0.050515,0.972950,0.012388,0.667968,0.094392
5,6,0.384284,0.009347,0.881487,0.028658,0.922690,0.021065,0.262407,0.021161,0.066292,...,0.470581,0.144105,0.887858,0.039124,0.796548,0.052702,0.974513,0.010351,0.658574,0.088853
6,7,0.397616,0.010964,0.882463,0.028429,0.925966,0.020493,0.261719,0.023040,0.066427,...,0.465353,0.105866,0.880734,0.050961,0.789048,0.057657,0.972301,0.014864,0.662463,0.124074
7,8,0.422033,0.016140,0.882377,0.028634,0.927537,0.019911,0.260795,0.023216,0.066359,...,0.446008,0.118061,0.877611,0.052244,0.786786,0.059490,0.971622,0.015063,0.670375,0.120265
8,9,0.444775,0.020913,0.882259,0.028586,0.928820,0.020135,0.260834,0.023461,0.066121,...,0.439978,0.115884,0.878818,0.052247,0.782619,0.057844,0.972094,0.014671,0.684119,0.128470
9,10,0.463789,0.023304,0.880956,0.029291,0.927338,0.020547,0.261691,0.024002,0.066382,...,0.445185,0.121051,0.874913,0.054241,0.784286,0.054180,0.971032,0.015096,0.669560,0.131644


In [10]:
import plotly.graph_objects as go

def line_with_band(metric):
    dfm = res.groupby("K")[metric].agg(["mean","std"]).reset_index().sort_values("K")
    dfm.columns = ["K","mean","std"]
    dfm["lo"] = dfm["mean"] - dfm["std"]
    dfm["hi"] = dfm["mean"] + dfm["std"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["mean"], mode="lines+markers", name=f"{metric} mean"))
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["hi"], mode="lines", line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["lo"], mode="lines", fill="tonexty",
                             line=dict(width=0), name="±1 std", opacity=0.2))
    fig.update_layout(
        title=f"{metric} vs K (mean ± std across repeats)",
        xaxis_title="K",
        yaxis_title=metric,
        height=520
    )
    fig.show()

for m in [
    "test_PR_AUC","test_ROC_AUC","test_LogLoss","test_Brier",
    "youden_test_Sensitivity","youden_test_Specificity","youden_test_Precision",
    "f1_test_Sensitivity","f1_test_Specificity","f1_test_Precision"
]:
    line_with_band(m)